# 00 — Setup and dataset primer

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/00_setup_and_data.ipynb)

**What this notebook is.** A one-page introduction to the AguaTrack
dataset and how to read its zarr archives directly with
`xarray.open_zarr` from HuggingFace. Run this first to confirm your
environment can talk to the data.

**What AguaTrack is.** AguaTrack v1 is the WAM2Layers (v3.3.1) backward
moisture-tracking output over South America at 0.25° daily resolution,
1990–2019. For any tagged grid cell on any day, the dataset answers:
*"the rain that fell here today — where did the water originally evaporate
from?"* The output variable `e_track[time, tagging_mask, latitude,
longitude]` is the spatial map of that source field, in mm/day of
water-equivalent.

**How to cite.** TODO: paper in submission. Once published, the BibTeX entry
will appear here.

**Tutorials in this repo.**

| # | Story | Region | Notebook |
|---|---|---|---|
| 01 | Serra do Mar 2011 flash-flood event | SE Brazil | `01_event_serra_do_mar_2011` |
| 02 | The "aerial river" feeding Santa Cruz | Bolivia <- Amazon | `02_aerial_river_santa_cruz` |
| 03 | ENSO-phase composites of moisture sources | Central Chile | `03_enso_composites` |
| 04 | The collapse of Aculeo Lake | Central Chile | `04_aculeo_lake_collapse` |
| 05 | Central-Chile megadrought decade | Central Chile | `05_megadrought_central_chile` |

## Step 1 — Configuration

Everything you might want to edit lives in this single cell.

AguaTrack is published on HuggingFace at
[`AguaTrack/AguaTrack-ARCO-SA`](https://huggingface.co/datasets/AguaTrack/AguaTrack-ARCO-SA),
with sub-directories for the daily store and (later) the monthly /
yearly aggregates. Throughout the tutorials we read these stores
directly with `xarray.open_zarr` over `fsspec`'s HuggingFace
protocol — no client library, no download step.

In [ ]:
HF_REVISION = "main"  # any branch / tag / commit on the HF repo
AGUATRACK_DAILY_2011 = (
    "hf://datasets/AguaTrack/AguaTrack-ARCO-SA"
    "/AguaTrack_ARCO_SA_daily/2011.zarr"
)

## Step 2 — Install dependencies (Colab only)

Local users have already run `uv sync` (or an equivalent
`pip install`). On Colab we install the geo stack that isn't
pre-bundled. The programmatic `get_ipython().run_line_magic("pip",
...)` form keeps the source `.py` syntactically valid Python.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri geopandas "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask pillow",
    )

## Step 3 — Open one year and print the schema

We open the 2011 daily store (the year used by the event notebook) and
print its `xarray.Dataset` summary. The read is lazy — no chunk bytes
are transferred until you slice and `.load()` / `.compute()`.

In [ ]:
import xarray as xr

ds = xr.open_zarr(AGUATRACK_DAILY_2011, storage_options={"revision": HF_REVISION})
print(ds)

## Step 4 — The schema in one page

Every annual AguaTrack zarr has the following layout:

```
Dimensions:        (time: 365, tagging_mask: 25186, latitude: 301, longitude: 261)
Coordinates:
  * time           datetime64[ns]  YYYY-01-01 ... YYYY-12-31
  * tagging_mask   int32           0 ... 25185
    tag_lat        (tagging_mask)  float32     latitude  of each tag cell
    tag_lon        (tagging_mask)  float32     longitude of each tag cell
  * latitude       (latitude)      float32     +15.0 -> -60.0   [DESCENDING]
  * longitude      (longitude)     float32     -90.0 -> -25.0   [ascending]
Data variables:
    e_track        (time, tagging_mask, latitude, longitude)  float32
    gains          (time, tagging_mask, latitude, longitude)  float32
    losses         (time, tagging_mask, latitude, longitude)  float32
    tagged_precip  (time, tagging_mask)                       float32
```

**Units.** All variables are stored as `kg m^-2` accumulated per output
time step (1 day). Numerically these are equivalent to **mm/day** of
water-equivalent.

**Variables.**

| Variable | Meaning |
|---|---|
| `e_track` | tracked evaporation. Moisture evaporated at `(lat, lon)` that ends up as rain at the tag cell on day `time`. **Primary variable** for the question "where did this rain come from?" |
| `gains` | moisture gained by the tracked parcel along its trajectory |
| `losses` | moisture lost by the tracked parcel along its trajectory |
| `tagged_precip` | precipitation at each tag cell per day. Useful for time-series and for picking event days |

## Step 5 — Five things to know before you slice

These five points cover almost every gotcha you will hit when working
with the dataset. Read them once and you will avoid most accidental
memory blow-ups.

### 1. `tagging_mask` is sparse coverage of the tracking domain

Each `tagging_mask` index corresponds to one specific grid cell. About
32% of the 78561 cells in the tracking domain are tagged (the
land + near-coastal portion). To find the tag for a real-world location,
search `tag_lat` / `tag_lon` for the nearest cell — every story notebook
does this with a simple `np.argmin`.

### 2. Slice tags first, time/space second

Chunks bundle `(full year, 10 tags, full map)`. So:

- YES: `ds.e_track.isel(tagging_mask=i)` reads one chunk (~1.15 GB)
- NO:  `ds.e_track.sel(time="2011-01-12")` touches every chunk in the year

### 3. Lazy by default — never `.load()` the full variable

A full 4-D variable is ~3 TB/year uncompressed float32. Always
`isel(tagging_mask=...)` or `isel(time=...)` first.

### 4. `latitude` is descending

`ds.latitude[0] = +15.0`, `ds.latitude[-1] = -60.0`. Slices must respect
this order:

```python
ds.sel(latitude=slice(0, -30))     # OK     0 deg N down to 30 deg S
ds.sel(latitude=slice(-30, 0))     # WRONG  returns empty
```

### 5. Backward tracking only

`e_track[t, tag, y, x]` answers *"where did this rain come from?"*. It
does not answer *"where did this evaporation go?"*. Forward tracking
would require a separate WAM2Layers run.

## Step 6 — Cheap diagnostic: the tagged-precipitation time series

The 1-D `tagged_precip[time, tag]` variable is the cheapest sanity
check we have. Per tag per year it is a few kB — basically free to
read. We use it below to pick the Serra do Mar tag (the one closest to
Nova Friburgo, Rio de Janeiro state) and print the first week of
January 2011. The peak day in the printed values matches the night of
11–12 January 2011 when the Serra do Mar landslides began.

In [ ]:
import numpy as np

# Pick the Serra do Mar tag (used by notebook 01) and print Jan 2011.
target_lat, target_lon = -22.4, -42.6

# Find the nearest tag in pure xarray. Squared Euclidean distance is
# fine at 0.25 deg resolution — no great-circle needed. `.argmin()`
# returns a 0-D DataArray; cast to int to use it as an index.
dist_sq = (ds.tag_lat - target_lat) ** 2 + (ds.tag_lon - target_lon) ** 2
i = int(dist_sq.argmin())

# 1-D `tagged_precip` slice for one tag and 31 days — only a few
# hundred floats. xarray handles the read lazily; printing or iterating
# triggers compute automatically.
tp = ds.tagged_precip.isel(tagging_mask=i).sel(time=slice("2011-01-01", "2011-01-31"))
for t, v in zip(tp.time.values[:7], tp.values[:7]):
    print(f"  {str(t)[:10]}  tagged_precip = {float(v):6.2f} mm/day")
print(f"  ... {tp.sizes['time']} days total, peak = {float(tp.max()):.1f} mm/day")

## Where to go next

Open one of the story notebooks listed at the top. Each one repeats
the same `xr.open_zarr("hf://datasets/AguaTrack/AguaTrack-ARCO-SA/...")`
pattern shown above, swapping the daily sub-directory for the monthly
or yearly aggregate as needed.